[![Kaggle](https://kaggle.com/static/images/open-in-kaggle.svg)](https://www.kaggle.com/code/maryrobynbernabe/kulintang-4-classification-model-development)

# Loading Training data

## Set the path to training dataset

In [ ]:
TRAIN_CSV = "/kaggle/input/notebooks/maryrobynbernabe/kulintang-3-train-test-split-csv/train_set.csv"

## Load and visualize the data

In [ ]:
import pandas as pd

# load the train csv
df_train = pd.read_csv(TRAIN_CSV)
df_train.head()

## Convert label column to numbers

In [ ]:
LABEL_COLUMN = "label"

print(df_train[LABEL_COLUMN].value_counts())

## Separate features and labels

In [ ]:
# gets all features without the labels and discrete features
cols_to_drop = ['video_segment_name', LABEL_COLUMN]
X_train = df_train.drop(columns=cols_to_drop, axis = 1).values

# all rows
# only the lastb column, which is the label
Y_train = df_train[LABEL_COLUMN].values

# Training the model using cross validation

## Function to perform 5-fold cross validation

In [ ]:
from sklearn.model_selection import KFold
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
import numpy as np

def cross_validation(model, data = (X_train, Y_train), splits = 5):
    """
    Performs k-fold cross-validation on a given model.
    :param model: The machine learning model to be evaluated.
    :param data: A tuple containing the features and labels.
    :param splits: The number of splits for k-fold cross-validation.
    :return: The mean accuracy of the model across all splits.
    """

    kf = KFold(n_splits=splits, shuffle=True, random_state=42)

    # Perform k-fold cross-validation
    accuracy = []
    precision = []
    recall = []
    f1 = []

    for train_index, valid_index in kf.split(data[0]):
        X_train, X_valid = data[0][train_index], data[0][valid_index]
        y_train, y_valid = data[1][train_index], data[1][valid_index]

        # Fit the defined model
        model.fit(X_train, y_train)

        # Make predictions on the test data
        y_pred = model.predict(X_valid)

        # Calculate accuracy, precision and recall
        accuracy.append(accuracy_score(y_valid, y_pred))
        precision.append(precision_score(y_valid, y_pred, average = 'weighted'))
        recall.append(recall_score(y_valid, y_pred, average = 'weighted'))
        f1.append(f1_score(y_valid, y_pred, average='weighted'))


    # get arrays
    accuracy_set = np.array(accuracy)
    precision_set = np.array(precision)
    recall_set = np.array(recall)
    f1_set = np.array(f1)

    # print("Mean Accuracy: {}".format(accuracy_set.mean()))
    # print("Mean Precision: {}".format(precision_set.mean()))
    # print("Mean Recall: {}".format(recall_set.mean()))
    # print("Mean F1-Score: {}".format(f1_set.mean()))
    return accuracy_set.mean()

## Function to evaluate the model

In [ ]:
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report
import matplotlib.pyplot as plt
import seaborn as sns
import numpy as np


def model_evaluations(y_true, y_pred):
    """
    Evaluate the performance of a classification model.
    :param y_true: The true labels.
    :param y_pred: The predicted labels.
    :return: The confusion matrix.
    """
    acc_score = accuracy_score(y_true, y_pred)
    print("Accuracy score: {:.4f}\n".format(acc_score))
    print("Classification Report:\n", classification_report(y_true, y_pred, target_names=labels))
    
    # Compute normalized confusion matrix
    cm = confusion_matrix(y_true, y_pred, normalize='true') * 100
    
    # Create annotations (only if >0, else empty string)
    annotations = np.array([[f"{val:.1f}%" if val > 0 else "" for val in row] for row in cm])
    
    # Plot
    plt.figure(figsize=(12, 10))  # bigger figure
    sns.heatmap(
        cm,
        annot=annotations,
        fmt="",
        cmap="Blues",
        xticklabels=labels,
        yticklabels=labels,
        cbar_kws={'label': 'Percentage (%)'}
    )

    plt.title("Normalized Confusion Matrix", fontsize=14, pad=20)
    plt.xlabel("Predicted Label", fontsize=12)
    plt.ylabel("True Label", fontsize=12)

    plt.xticks(rotation=45, ha="right", fontsize=9)  # rotated x labels
    plt.yticks(rotation=0, fontsize=9)

    plt.tight_layout()
    plt.show()
    return cm


## Training a KNN model

### Evaluating KNN algorithm for different values of k

In [ ]:
from sklearn.neighbors import KNeighborsClassifier
KNN_Accuracies = []
for i in range (1,20):
  knn = KNeighborsClassifier(n_neighbors=i) # vary n_neighbours from 1 to 20
  KNN_Accuracies.append(cross_validation(knn))


### Print accuracies

In [ ]:
for i in KNN_Accuracies:
  print(i)

## Training a Random Forest model

### Evaluating RandomForest algorithm for various values of estimators and depth

In [ ]:
from sklearn.ensemble import RandomForestClassifier
import numpy as np

n_estimators = np.arange(10, 101, 10)
max_depths = np.arange(1, 15, 1)

best_accuracy = []

for i in n_estimators:
  temp = []
  for j in max_depths:
    rf = RandomForestClassifier(n_estimators=i, max_depth = j)
    temp.append(cross_validation(rf))
  best_accuracy.append(temp)

### Print accuracies

In [ ]:
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

## Training an MLP model

### Evaluating MLP Architecture for various values of epochs and learning rates

In [ ]:
import numpy as np
from sklearn.neural_network import MLPClassifier

# Use 5-fold cross validation for hyper-parameter tuning
# Try out different values and choose the best hyper-parameters

learning_rate = [0.01, 0.005, 0.001, 0.0005, 0.0001]
epoch = np.arange(10, 210, 20)

best_accuracy = []

for i in epoch:
  temp = []
  for j in learning_rate:
    mlp = MLPClassifier(hidden_layer_sizes=(100,500), learning_rate_init=0.001, max_iter=20) # vary learning rate as 0.0001, 0.001, 0.01, 0.05, 0.1, 1 and max_iter from 10 to 100
    temp.append(cross_validation(mlp))
  best_accuracy.append(temp)

### Print accuracies

In [ ]:
for row in best_accuracy:
  print("\t".join(map(str, row))) # Convert each element to string and join with tabs

## Train the best model

In [ ]:
from sklearn.ensemble import RandomForestClassifier
rf_best_model = RandomForestClassifier(n_estimators=50, max_depth = 10)
rf_best_model.fit(X_train, Y_train)

### Saving the best model

In [ ]:
import pickle
fh = open("rf_best_model.pkl", "wb")
pickle.dump(rf_best_model, fh)
fh.close()